# Migration Secrets Setup

This notebook stores PAT tokens for cross-workspace dashboard migration.

**What this does:**
- Creates secret scope `migration_secrets` (if doesn't exist)
- Stores target workspace PAT token
- Tests connection to both workspaces

**Run once, then use for all migration jobs.**

**⚠️ Keep this notebook PRIVATE - contains credential setup!**

---

## Cell 1: Create Secret Scope

In [ ]:
# Check if secret scope exists
print("="*80)
print("CHECKING SECRET SCOPE")
print("="*80)

print("\nℹ️  Secret scopes must be created via CLI (not via dbutils)\n")

try:
    # Try to list the scope (will fail if doesn't exist)
    secrets = dbutils.secrets.list("migration_secrets")
    print("✅ Secret scope 'migration_secrets' already exists")
    print(f"   Secrets stored: {len(secrets)}")
except Exception as e:
    print("❌ Secret scope 'migration_secrets' does NOT exist\n")
    print("📋 To create it, run this CLI command from your terminal:\n")
    print("   databricks secrets create-scope migration_secrets --profile source-workspace\n")
    print("Then come back and re-run this cell.")
    raise

print("\n" + "="*80)

## Alternative: Create Secret Scope via CLI

If the cell above failed, run this command in your terminal:

```bash
databricks secrets create-scope migration_secrets --profile source-workspace
```

Then re-run Cell 1 to verify.

## Cell 2: Store Target Workspace PAT Token

In [ ]:
# INSTRUCTIONS:
# 1. Open TARGET workspace: https://fevm-akrishn-stable-classic-vv5y0k.cloud.databricks.com
# 2. Go to: User Settings → Developer → Access Tokens
# 3. Click: Generate New Token
# 4. Set Comment: "Dashboard Migration" and Lifetime: 90 days
# 5. Click: Generate
# 6. Copy the token (you won't see it again!)
# 7. Run this cell and paste when prompted

print("="*80)
print("STORING TARGET WORKSPACE PAT TOKEN")
print("="*80)
print("\n📋 Target Workspace: https://fevm-akrishn-stable-classic-vv5y0k.cloud.databricks.com\n")
print("⚠️  You'll be prompted to enter the token in a separate input box")
print("   (Token will not be displayed for security)\n")

dbutils.secrets.put("migration_secrets", "target_workspace_token")

print("\n✅ Target workspace PAT token stored successfully!")
print("\n" + "="*80)

## Cell 3: Verify Secrets Stored

In [ ]:
# List stored secrets (values are hidden for security)
print("="*80)
print("SECRETS VERIFICATION")
print("="*80)

secrets = dbutils.secrets.list("migration_secrets")

print("\n📋 Secrets stored in 'migration_secrets' scope:\n")
for secret in secrets:
    print(f"   ✅ {secret.key}")

print(f"\nℹ️  Total secrets: {len(secrets)}")
print("\n" + "="*80)

## Cell 4: Test Connection to Target Workspace

In [ ]:
# Test that the stored token works and connects to the CORRECT workspace
from databricks.sdk import WorkspaceClient

print("="*80)
print("TESTING TARGET WORKSPACE CONNECTION")
print("="*80)

target_url = "https://fevm-akrishn-stable-classic-vv5y0k.cloud.databricks.com"
print(f"\n🔍 Testing connection to: {target_url}\n")

try:
    # Get token from secrets
    token = dbutils.secrets.get("migration_secrets", "target_workspace_token")
    print("✅ Token retrieved from secrets")
    
    # Create client with explicit host and token
    client = WorkspaceClient(host=target_url, token=token)
    print("✅ Client created")
    
    # Test connection
    user = client.current_user.me()
    print("✅ Connection successful!\n")
    
    print(f"📊 Connection Details:")
    print(f"   Client host: {client.config.host}")
    print(f"   Expected host: {target_url}")
    print(f"   Authenticated as: {user.user_name}")
    print(f"   User ID: {user.id}")
    
    # Verify it's the TARGET workspace (fevm-akrishn)
    client_host_clean = client.config.host.rstrip('/')
    target_url_clean = target_url.rstrip('/')
    
    print(f"\n🔍 Workspace Verification:")
    if "fevm-akrishn" in client.config.host:
        print(f"   ✅ VERIFIED: Connected to TARGET workspace (fevm-akrishn)!")
        print(f"   ✅ Dashboards will be created in the CORRECT workspace")
    else:
        print(f"   ❌ WARNING: Connected to unexpected workspace!")
        print(f"   Expected: fevm-akrishn")
        print(f"   Got: {client.config.host}")
    
    if client_host_clean == target_url_clean:
        print(f"   ✅ Host URLs match exactly")
    else:
        print(f"   ⚠️  Host URLs don't match:")
        print(f"      Client: {client_host_clean}")
        print(f"      Expected: {target_url_clean}")
    
    # Test a simple API call
    print(f"\n🧪 Testing API access...")
    warehouses = list(client.warehouses.list())
    print(f"   ✅ Can list warehouses: {len(warehouses)} found")
    
    print("\n" + "="*80)
    print("CONNECTION TEST PASSED")
    print("="*80)
    print("\n✅ Ready to run migration!")
    print("\n📋 Next command:")
    print("   databricks bundle deploy -t dev-live --profile source-workspace")
    print("   databricks bundle run generate_deploy -t dev-live --profile source-workspace")
    
except Exception as e:
    print(f"\n❌ CONNECTION FAILED: {e}")
    print("\n🔧 Troubleshooting:")
    print("   1. Verify token is valid (check expiration in target workspace)")
    print("   2. Verify token is from TARGET workspace (fevm-akrishn)")
    print("   3. Regenerate token if expired or incorrect")
    print("   4. Re-run Cell 2 to store new token")
    raise

print("="*80)

## Cell 5: Optional - Store Service Principal Credentials (For Future)

In [ ]:
# OPTIONAL: Store Service Principal credentials (when you have Account Console access)
# Service Principal is more secure than PAT token

print("="*80)
print("SERVICE PRINCIPAL SETUP (OPTIONAL)")
print("="*80)

print("\n📋 Instructions:")
print("   1. Create SP in Account Console → Service Principals")
print("   2. Generate client secret (save it!)")
print("   3. Note the Application (Client) ID (UUID)")
print("   4. Assign SP to both workspaces in Account Console")
print("   5. Uncomment and run the code below\n")

# UNCOMMENT TO USE:
# print("Storing SP client_id...")
# sp_client_id = "10eac1b8-b722-4928-9c27-72c6d37e21ff"  # Replace with your SP UUID
# dbutils.secrets.put("migration_secrets", "sp_client_id", sp_client_id)
# print("✅ SP client_id stored")

# print("\nStoring SP client_secret...")
# dbutils.secrets.put("migration_secrets", "sp_secret")
# print("✅ SP client_secret stored")

print("\n⚠️  Service Principal setup is optional")
print("   PAT token will work for now")
print("   Switch to SP later for production use")
print("\n" + "="*80)

---

## ✅ Setup Complete

Run Cells 1-4 to complete setup.

**Security Best Practices:**
1. ✅ Use Service Principal (when available) - Most secure
2. ⚠️ Use PAT Token (current) - Quick but less secure
3. ❌ Avoid hardcoding tokens in code

**This notebook should be:**
- Kept private (only you can access)
- Run from SOURCE workspace (e2-demo-field-eng)
- Run once per token rotation (90 days for PAT)